# Fitness Coach — RunPod Training (Qwen2.5-7B, Unsloth)

**Before running:** in the Jupyter file browser (left), drag in the 3 data files
`sft.jsonl`, `dpo.jsonl`, `eval.jsonl` so they sit next to this notebook.
Then run every cell top-to-bottom. Stop/terminate the pod when finished.

## 1. Check GPU + data files

In [1]:
import torch, os
print("GPU:", torch.cuda.get_device_name(0))
print("jsonl files here:", [f for f in os.listdir(".") if f.endswith(".jsonl")])
assert torch.cuda.is_available(), "No GPU on this pod."

GPU: NVIDIA GeForce RTX 3090
jsonl files here: ['dpo.jsonl', 'sft.jsonl', 'eval.jsonl']


## 2. Install Unsloth (skip if the pod template already has it)

In [2]:
!pip install -q unsloth
print("ready")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
ready


## 3. SFT training (stage 1, ~15-25 min)

In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAXLEN = 4096

model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAXLEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

ds = load_dataset("json", data_files="sft.jsonl", split="train")
ds = ds.map(lambda ex: {"text": tok.apply_chat_template(ex["messages"], tokenize=False)})

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAXLEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4, logging_steps=10,
        output_dir="outputs/sft", optim="adamw_8bit"))
trainer.train()
model.save_pretrained("outputs/sft-adapter"); tok.save_pretrained("outputs/sft-adapter")
print("SFT done")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 600 | Num Epochs = 2 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.921252
20,0.513685
30,0.212108
40,0.163034
50,0.139037
60,0.117248
70,0.111043
80,0.114184
90,0.103474
100,0.098249


Unsloth: Restored added_tokens_decoder metadata in outputs/sft/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/sft-adapter/tokenizer_config.json.


SFT done


## 4. DPO training (stage 2, ~10-20 min)

In [6]:
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

dpo_ds = load_dataset("json", data_files="dpo.jsonl", split="train")

trainer = DPOTrainer(
    model=model, tokenizer=tok, train_dataset=dpo_ds,
    args=DPOConfig(beta=0.1, per_device_train_batch_size=1,
        gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=5e-5,
        logging_steps=10, output_dir="outputs/dpo", optim="adamw_8bit",
        max_length=4096, max_prompt_length=1024))
trainer.train()
model.save_pretrained("outputs/dpo-adapter"); tok.save_pretrained("outputs/dpo-adapter")
print("DPO done")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.000000,52.684105,28.308142,1.000000,24.375957,-373.696350,-626.555115,-1.692561,-1.590558
20,0.000000,53.184856,26.686285,1.000000,26.498571,-365.782623,-634.527222,-1.681137,-1.583025
30,0.000000,52.060081,22.477695,1.000000,29.582378,-364.830658,-662.716431,-1.683083,-1.587570
40,0.000000,52.615070,22.904860,1.000000,29.710211,-360.894958,-662.932983,-1.689415,-1.585814
50,0.000000,52.573547,22.500370,1.000000,30.073177,-362.384430,-669.732788,-1.675902,-1.578176


DPO done


## 5. Quick test — plan for an injured user

In [7]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"35yo, 80kg, lose fat, home dumbbells, knee pain, 3 days/week. Return the JSON plan."}]
ids = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(input_ids=ids, max_new_tokens=1200, temperature=0.7)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.11/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

{"day1": {"calories": 2070, "protein_g": 144, "carbs_g": 236, "fat_g": 79, "workouts": [{"exercise": "One-Arm Kettlebell Push Press", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds shoulders."}, {"exercise": "Seated Dumbbell Palms-Up Wrist Curl", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds forearms."}, {"exercise": "Dumbbell One-Arm Shoulder Press", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds shoulders."}, {"exercise": "Pushups (Close and Wide Hand Positions)", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds chest."}]}, "day2": {"calories": 2270, "protein_g": 144, "carbs_g": 236, "fat_g": 79, "workouts": [{"exercise": "Dumbbell Seated Box Jump", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds quadriceps."}, {"exercise": "Standing Bent-Over Two-Arm Dumbbell Triceps Extension", "sets": 3, "reps": "10-15", "rest_seconds": 90, "why": "Builds triceps."}, {"exercise": "Cable Shoulder Press", "sets": 3, "reps"

## 6. Evaluate the fine-tuned model (resume numbers)

In [8]:
import json
CONTRA = {"knee pain":["squat","lunge","leg press"],
          "shoulder impingement":["overhead press","upright row","bench press"],
          "lower back pain":["deadlift","good morning","bent over row"]}

def extract_json(t):
    try: return json.loads(t)
    except Exception: pass
    i, j = t.find("{"), t.rfind("}")
    if i >= 0 and j > i:
        try: return json.loads(t[i:j+1])
        except Exception: return None
    return None

def avoids_injury(plan, injury):
    if not injury or injury == "None": return True
    bad = CONTRA.get(injury, [])
    for d in plan.get("weekly_workouts", []):
        for e in d.get("exercises", []):
            if any(b in e.get("name","").lower() for b in bad): return False
    return True

def respects_equipment(plan, eq):
    if eq != "bodyweight only": return True
    banned = ["barbell","machine","cable","leg press"]
    for d in plan.get("weekly_workouts", []):
        for e in d.get("exercises", []):
            if any(b in e.get("name","").lower() for b in banned): return False
    return True

def gen(p):
    u = (f"Profile: {p['age']}yo, {p['weight_kg']}kg, goal: {p['goal']}, "
         f"equipment: {p['equipment']}, diet: {p['diet']}, experience: {p['experience']}, "
         f"injury: {p['injury']}, {p['days_per_week']} days/week. Return the JSON plan.")
    ids = tok.apply_chat_template([{"role":"user","content":u}], return_tensors="pt",
                                  add_generation_prompt=True).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=1200, temperature=0.7)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

rows = [json.loads(l) for l in open("eval.jsonl")][:100]
valid = eq = inj = 0
for r in rows:
    p = r["profile"]; plan = extract_json(gen(p))
    if plan is None: continue
    valid += 1
    eq += int(respects_equipment(plan, p["equipment"]))
    inj += int(avoids_injury(plan, p["injury"]))
n = len(rows)
print(f"Fine-tuned Qwen2.5-7B on {n} held-out profiles:")
print(f"  valid JSON:             {valid}/{n} = {valid/n:.0%}")
print(f"  equipment satisfaction: {eq}/{n} = {eq/n:.0%}")
print(f"  injury safety:          {inj}/{n} = {inj/n:.0%}")
print("\nPaste these to Claude to fill docs/RESULTS.md")

Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

AttributeError: 'str' object has no attribute 'get'

In [11]:
import warnings, logging, json
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

CONTRA = {"knee pain":["squat","lunge","leg press"],
          "shoulder impingement":["overhead press","upright row","bench press"],
          "lower back pain":["deadlift","good morning","bent over row"]}

def extract_json(t):
    try: return json.loads(t)
    except Exception: pass
    i, j = t.find("{"), t.rfind("}")
    if i >= 0 and j > i:
        try: return json.loads(t[i:j+1])
        except Exception: return None
    return None

def exercise_names(plan):
    names = []
    wk = plan.get("weekly_workouts")
    if isinstance(wk, list):
        for d in wk:
            if isinstance(d, dict):
                for e in d.get("exercises", []):
                    if isinstance(e, dict): names.append(str(e.get("name","")))
                    elif isinstance(e, str): names.append(e)
    for v in plan.values():
        if isinstance(v, dict) and isinstance(v.get("workouts"), list):
            for e in v["workouts"]:
                if isinstance(e, dict): names.append(str(e.get("exercise") or e.get("name","")))
    return [n.lower() for n in names if n]

def schema_ok(plan):
    wk = plan.get("weekly_workouts")
    if not (isinstance(wk, list) and wk and isinstance(wk[0], dict)): return False
    exs = wk[0].get("exercises")
    return isinstance(exs, list) and all(isinstance(e, dict) and "name" in e for e in exs) and "nutrition" in plan

def avoids_injury(names, injury):
    if not injury or injury == "None": return True
    return not any(b in n for n in names for b in CONTRA.get(injury, []))

def respects_equipment(names, eq):
    if eq != "bodyweight only": return True
    banned = ["barbell","machine","cable","leg press"]
    return not any(b in n for n in names for b in banned)

def gen(p):
    u = (f"Profile: {p['age']}yo, {p['weight_kg']}kg, goal: {p['goal']}, "
         f"equipment: {p['equipment']}, diet: {p['diet']}, experience: {p['experience']}, "
         f"injury: {p['injury']}, {p['days_per_week']} days/week. Return the JSON plan.")
    ids = tok.apply_chat_template([{"role":"user","content":u}], return_tensors="pt",
                                  add_generation_prompt=True).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=1200, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

rows = [json.loads(l) for l in open("eval.jsonl")][:40]
valid = schema = eq = inj = 0
for k, r in enumerate(rows, 1):
    p = r["profile"]; plan = extract_json(gen(p))
    if plan is not None:
        valid += 1
        if schema_ok(plan): schema += 1
        names = exercise_names(plan)
        eq += int(respects_equipment(names, p["equipment"]))
        inj += int(avoids_injury(names, p["injury"]))
    if k % 10 == 0: print(f"...{k}/{len(rows)} done")
n = len(rows)
print(f"\nFine-tuned Qwen2.5-7B on {n} held-out profiles:")
print(f"  valid JSON:             {valid}/{n} = {valid/n:.0%}")
print(f"  schema match:           {schema}/{n} = {schema/n:.0%}")
print(f"  equipment satisfaction: {eq}/{n} = {eq/n:.0%}")
print(f"  injury safety:          {inj}/{n} = {inj/n:.0%}")


...10/40 done
...20/40 done
...30/40 done
...40/40 done

Fine-tuned Qwen2.5-7B on 40 held-out profiles:
  valid JSON:             26/40 = 65%
  schema match:           0/40 = 0%
  equipment satisfaction: 26/40 = 65%
  injury safety:          26/40 = 65%


## 7. Save the model

In [13]:
import shutil
shutil.make_archive("dpo-adapter", "zip", "outputs/dpo-adapter")
shutil.make_archive("sft-adapter", "zip", "outputs/sft-adapter")
print("Done — download both .zip files from the file browser (right-click → Download).")


Done — download both .zip files from the file browser (right-click → Download).


In [14]:
import torch, warnings, logging, json
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
del model; torch.cuda.empty_cache()
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("outputs/sft-adapter", max_seq_length=4096, load_in_4bit=True)
FastLanguageModel.for_inference(model)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

rows = [json.loads(l) for l in open("eval.jsonl")][:40]
valid = schema = eq = inj = 0
for k, r in enumerate(rows, 1):
    p = r["profile"]; plan = extract_json(gen(p))
    if plan is not None:
        valid += 1
        if schema_ok(plan): schema += 1
        names = exercise_names(plan)
        eq += int(respects_equipment(names, p["equipment"]))
        inj += int(avoids_injury(names, p["injury"]))
    if k % 10 == 0: print(f"...{k}/{len(rows)}")
n = len(rows)
print(f"\nSFT-ONLY model on {n} profiles:")
print(f"  valid JSON: {valid/n:.0%}  |  schema: {schema/n:.0%}  |  equip: {eq/n:.0%}  |  injury: {inj/n:.0%}")


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
...10/40
...20/40
...30/40
...40/40

SFT-ONLY model on 40 profiles:
  valid JSON: 48%  |  schema: 0%  |  equip: 48%  |  injury: 48%


In [12]:
!zip -qr dpo-adapter.zip outputs/dpo-adapter
print("Right-click dpo-adapter.zip in the file browser -> Download.")
# Optional: push to Hugging Face
# from huggingface_hub import login; login()
# model.push_to_hub("your-username/fitness-coach-qwen2.5-7b")

/bin/bash: line 1: zip: command not found
Right-click dpo-adapter.zip in the file browser -> Download.


In [15]:
SYSTEM = ("You are an expert fitness coach. Given a user profile, output ONLY a JSON "
    "object matching this schema: goal, experience (beginner|intermediate|advanced), "
    "daily_schedule{wake, workout{time,type}, meals[{time,name,focus}], sleep{target,hours}}, "
    "weekly_workouts[{day, focus, exercises[{name,sets,reps,rest_seconds,demo_image,why}]}], "
    "nutrition{daily_macros{calories,protein_g,carbs_g,fat_g}, "
    "example_day[{food,grams,calories,protein_g}], grocery_list[]}, disclaimer. "
    "Use realistic exercises and macros. Set demo_image to null (filled later). "
    "For beginners, fill 'why' with a short reason. Always include a disclaimer that "
    "this is general guidance, not medical advice. If the user reports an injury, "
    "avoid contraindicated movements.")

def gen(p):
    u = (f"Profile: {p['age']}yo, {p['weight_kg']}kg, goal: {p['goal']}, "
         f"equipment: {p['equipment']}, diet: {p['diet']}, experience: {p['experience']}, "
         f"injury: {p['injury']}, {p['days_per_week']} days/week. Return the JSON plan.")
    msgs = [{"role":"system","content":SYSTEM}, {"role":"user","content":u}]
    ids = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=1200, temperature=0.7, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

# quick single check:
p = {"age":35,"weight_kg":80,"goal":"lose fat","equipment":"home dumbbells",
     "diet":"no restriction","experience":"beginner","injury":"knee pain","days_per_week":3}
print(gen(p)[:700])


{"goal": "lose fat", "experience": "beginner", "daily_schedule": {"wake": "0000", "workout": {"time": "18:00", "type": "Full body"}, "meals": [{"time": "07:00", "name": "Breakfast", "focus": "protein + carbs"}, {"time": "13:00", "name": "Lunch", "focus": "balanced"}, {"time": "16:30", "name": "Snack", "focus": "pre-workout fuel"}, {"time": "22:00", "name": "Dinner", "focus": "protein + recovery"}], "sleep": {"target": "20:00-07:00", "hours": 8}}, "weekly_workouts": [{"day": "Monday", "focus": "Full body", "exercises": [{"name": "Kettlebell Hang Clean", "sets": 3, "reps": "10-15", "rest_seconds": 90, "demo_image": null, "why": "Builds quadriceps."}, {"name": "Standing Long Jump", "sets": 3, "


In [18]:
import json
rows = [json.loads(l) for l in open("eval.jsonl")][:40]
valid = schema = eq = inj = 0
for k, r in enumerate(rows, 1):
    p = r["profile"]; plan = extract_json(gen(p))
    if plan is not None:
        valid += 1
        if schema_ok(plan): schema += 1
        names = exercise_names(plan)
        eq += int(respects_equipment(names, p["equipment"]))
        inj += int(avoids_injury(names, p["injury"]))
    if k % 10 == 0: print(f"...{k}/{len(rows)}")
n = len(rows)
print(f"\nSFT model (WITH system prompt) on {n} profiles:")
print(f"  valid JSON: {valid/n:.0%}  |  schema: {schema/n:.0%}  |  equip: {eq/n:.0%}  |  injury: {inj/n:.0%}")


...10/40
...20/40
...30/40
...40/40

SFT model (WITH system prompt) on 40 profiles:
  valid JSON: 38%  |  schema: 38%  |  equip: 38%  |  injury: 35%
